# Parte 3 — Ecuaciones elípticas
## 3.9 Problemas de valores propios y armónicos esféricos
### 3.9.01 Espectro de Dirichlet, Laplace–Beltrami, separación esférica y resonancia

Este notebook cubre el apartado **3.9** del temario oficial:

> Problemas de valores propios (armónicos esféricos).

## Relación con las notas manuscritas

Las notas disponibles terminan en la página 9 con función de Green. No contienen
un bloque manuscrito sobre valores propios o armónicos esféricos. Por tanto, esta
unidad se marca íntegramente como

**Complemento para cerrar el temario oficial.**

Se conservan las convenciones ya utilizadas:

$$
-\Delta u=\lambda u
$$

para el problema espectral de Dirichlet, y

$$
-\Delta_{\mathbb S^{n-1}}Y=\mu Y
$$

para el operador de Laplace–Beltrami.

## Contenido

1. problema espectral de Dirichlet;
2. cociente de Rayleigh y primer valor propio;
3. ortogonalidad;
4. Laplaciano en coordenadas esféricas;
5. Laplace–Beltrami;
6. armónicos esféricos y primeros modos;
7. separación radial en la bola;
8. valores propios de la bola;
9. resonancia y condición de compatibilidad.

## Estado de las fuentes

Las capturas antes enlazadas de las notas, del temario y del Examen General 2025-1 no están incluidas en el repositorio. El desarrollo que sigue es autocontenido. Para cotejar el enunciado oficial debe consultarse el PDF original fuera de este repositorio; no se sustituye aquí por una imagen inventada.


# Simulaciones y visualizaciones

Las celdas se ejecutan directamente. No existe una bandera `VIDEO=True`.

Se incluyen:

1. superficies tridimensionales de armónicos esféricos reales;
2. animación rotatoria de un modo;
3. verificación numérica de ortogonalidad;
4. espectro de Dirichlet de la bola unitaria;
5. perfiles radiales mediante funciones de Bessel esféricas;
6. amplificación resonante cerca de un valor propio.

Las funciones especiales se evalúan con SciPy. CuPy/CUDA se utiliza
automáticamente para álgebra matricial densa cuando aporta valor.

In [ ]:
from __future__ import annotations

import math
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from IPython.display import Video, Image, display

from scipy.optimize import brentq
from scipy.special import spherical_jn

try:
    from scipy.special import sph_harm_y

    def complex_spherical_harmonic(l, m, theta, phi):
        """
        theta: ángulo polar en [0,pi]
        phi: ángulo azimutal en [0,2pi)
        """
        return sph_harm_y(l, m, theta, phi)

except ImportError:
    from scipy.special import sph_harm

    def complex_spherical_harmonic(l, m, theta, phi):
        return sph_harm(m, l, phi, theta)


BASE = Path("..")
FIG_DIR = BASE / "figuras"
ANIM_DIR = BASE / "animaciones"
FIG_DIR.mkdir(parents=True, exist_ok=True)
ANIM_DIR.mkdir(parents=True, exist_ok=True)

GPU_AVAILABLE = False

try:
    import cupy as cp

    if cp.cuda.runtime.getDeviceCount() > 0:
        GPU_AVAILABLE = True
        print("CuPy/CUDA disponible para álgebra densa.")
    else:
        raise RuntimeError("No se encontró un dispositivo CUDA.")
except Exception as exc:
    cp = None
    print("Álgebra densa: NumPy/CPU")
    print("CuPy no disponible:", type(exc).__name__)


def save_and_display_animation(
    animation,
    stem,
    fps=60,
    dpi=150,
    bitrate=10000,
):
    """Guarda y muestra automáticamente una animación."""
    if shutil.which("ffmpeg"):
        output = ANIM_DIR / f"{stem}.mp4"
        writer = FFMpegWriter(
            fps=fps,
            bitrate=bitrate,
            metadata={"title": stem},
        )
        animation.save(output, writer=writer, dpi=dpi)
        display(Video(str(output), embed=True))
    else:
        output = ANIM_DIR / f"{stem}.gif"
        writer = PillowWriter(fps=min(fps, 35))
        animation.save(
            output,
            writer=writer,
            dpi=min(dpi, 110),
        )
        display(Image(filename=str(output)))

    print("Animación guardada en:", output.resolve())
    return output


def real_spherical_harmonic(l, m, theta, phi):
    """
    Base real ortonormal obtenida a partir de los armónicos complejos.
    """
    if m == 0:
        return np.real(
            complex_spherical_harmonic(l, 0, theta, phi)
        )

    if m > 0:
        harmonic = complex_spherical_harmonic(
            l,
            m,
            theta,
            phi,
        )
        return (
            math.sqrt(2.0)
            * ((-1) ** m)
            * np.real(harmonic)
        )

    harmonic = complex_spherical_harmonic(
        l,
        -m,
        theta,
        phi,
    )
    return (
        math.sqrt(2.0)
        * ((-1) ** abs(m))
        * np.imag(harmonic)
    )


def positive_spherical_bessel_zeros(l, count):
    """Primeros ceros positivos de j_l."""
    zeros = []
    left = 1e-7
    step = math.pi / 18.0
    right = left + step
    value_left = spherical_jn(l, left)

    while len(zeros) < count:
        value_right = spherical_jn(l, right)

        if value_left * value_right < 0.0:
            root = brentq(
                lambda z: spherical_jn(l, z),
                left,
                right,
            )

            if not zeros or abs(root - zeros[-1]) > 1e-7:
                zeros.append(root)

        left = right
        value_left = value_right
        right = right + step

        if right > 300.0:
            raise RuntimeError(
                f"No se encontraron {count} ceros para l={l}."
            )

    return np.asarray(zeros)

## Simulación 3.9.A — Primeros armónicos esféricos reales

Para cada par $(\ell,m)$ se dibuja la superficie radial

$$
r(\theta,\varphi)
=
1+\alpha
\frac{Y_{\ell m}(\theta,\varphi)}
{\|Y_{\ell m}\|_\infty}.
$$

La deformación radial permite ver signos, lóbulos y conjuntos nodales.

In [ ]:
# ============================================================
# SUPERFICIES DE ARMÓNICOS ESFÉRICOS
# ============================================================

n_theta = 180
n_phi = 360

theta = np.linspace(0.0, math.pi, n_theta)
phi = np.linspace(
    0.0,
    2.0 * math.pi,
    n_phi,
    endpoint=False,
)

THETA, PHI = np.meshgrid(
    theta,
    phi,
    indexing="ij",
)

modes_to_plot = [
    (0, 0),
    (1, 0),
    (1, 1),
    (2, 0),
    (2, 2),
    (3, 2),
]

for l, m in modes_to_plot:
    harmonic = real_spherical_harmonic(
        l,
        m,
        THETA,
        PHI,
    )

    normalized = harmonic / np.max(
        np.abs(harmonic)
    )
    radius = 1.0 + 0.62 * normalized

    X = radius * np.sin(THETA) * np.cos(PHI)
    Y = radius * np.sin(THETA) * np.sin(PHI)
    Z = radius * np.cos(THETA)

    fig = plt.figure(figsize=(8.0, 7.0))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(
        X,
        Y,
        Z,
        linewidth=0,
        antialiased=True,
    )
    ax.set_box_aspect((1.0, 1.0, 1.0))
    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$y$")
    ax.set_zlabel(r"$z$")
    ax.set_title(
        rf"Armónico esférico real $(\ell,m)=({l},{m})$"
    )
    ax.view_init(elev=28, azim=34)
    fig.tight_layout()

    path = (
        FIG_DIR
        / f"03.9.A_armonico_l{l}_m{m}.png"
    )
    fig.savefig(path, dpi=220)
    plt.show()
    plt.close(fig)

    print(path.resolve())

## Simulación 3.9.B — Animación rotatoria del modo $(\ell,m)=(3,2)$

La superficie permanece fija y la cámara gira. La animación se genera y se
muestra automáticamente.

In [ ]:
l_animation = 3
m_animation = 2

harmonic = real_spherical_harmonic(
    l_animation,
    m_animation,
    THETA,
    PHI,
)
normalized = harmonic / np.max(np.abs(harmonic))
radius = 1.0 + 0.62 * normalized

X_animation = radius * np.sin(THETA) * np.cos(PHI)
Y_animation = radius * np.sin(THETA) * np.sin(PHI)
Z_animation = radius * np.cos(THETA)

fig = plt.figure(figsize=(8.0, 7.0))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(
    X_animation,
    Y_animation,
    Z_animation,
    linewidth=0,
    antialiased=True,
)
ax.set_box_aspect((1.0, 1.0, 1.0))
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_zlabel(r"$z$")
ax.set_title(
    r"Rotación del modo real $(\ell,m)=(3,2)$"
)

azimuths = np.linspace(0.0, 360.0, 180)


def update_rotation(frame):
    ax.view_init(
        elev=28.0,
        azim=float(azimuths[frame]),
    )
    return (ax,)


animation = FuncAnimation(
    fig,
    update_rotation,
    frames=len(azimuths),
    interval=1000.0 / 60.0,
    blit=False,
)

fig.tight_layout()

save_and_display_animation(
    animation,
    "03.9.B_rotacion_armonico_l3_m2",
    fps=60,
    dpi=165,
    bitrate=14000,
)

plt.close(fig)

## Simulación 3.9.C — Ortogonalidad numérica

Se calculan los productos internos

$$
\int_{\mathbb S^2}
Y_{\ell m}Y_{\ell' m'}\,dS
$$

para todos los modos con $0\leq\ell\leq4$.

La matriz de Gram debe aproximar la identidad.

In [ ]:
# ============================================================
# MATRIZ DE GRAM
# ============================================================

n_mu = 150
n_phi_quad = 320

mu_nodes, mu_weights = np.polynomial.legendre.leggauss(n_mu)
theta_quad = np.arccos(mu_nodes)
phi_quad = np.linspace(
    0.0,
    2.0 * math.pi,
    n_phi_quad,
    endpoint=False,
)

THETA_Q, PHI_Q = np.meshgrid(
    theta_quad,
    phi_quad,
    indexing="ij",
)

modes = [
    (l, m)
    for l in range(5)
    for m in range(-l, l + 1)
]

values = []

for l, m in modes:
    values.append(
        real_spherical_harmonic(
            l,
            m,
            THETA_Q,
            PHI_Q,
        ).reshape(n_mu, n_phi_quad)
    )

values = np.asarray(values)
phi_weight = 2.0 * math.pi / n_phi_quad

if GPU_AVAILABLE:
    values_gpu = cp.asarray(values)
    weights_gpu = cp.asarray(
        mu_weights[:, None]
    )

    weighted_values = (
        values_gpu
        * cp.sqrt(weights_gpu * phi_weight)
    ).reshape(len(modes), -1)

    gram = cp.asnumpy(
        weighted_values
        @ weighted_values.T
    )
else:
    weighted_values = (
        values
        * np.sqrt(
            mu_weights[:, None] * phi_weight
        )
    ).reshape(len(modes), -1)

    gram = weighted_values @ weighted_values.T

identity_error = gram - np.eye(len(modes))

fig, ax = plt.subplots(figsize=(8.0, 7.0))
image = ax.imshow(identity_error)
fig.colorbar(
    image,
    ax=ax,
    label=r"$G-I$",
)
ax.set_xlabel("índice del modo")
ax.set_ylabel("índice del modo")
ax.set_title("Error de la matriz de Gram")
fig.tight_layout()

path = FIG_DIR / "03.9.C_ortogonalidad_gram.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print(
    "Máximo error absoluto:",
    float(np.max(np.abs(identity_error))),
)
print(path.resolve())

## Simulación 3.9.D — Espectro de Dirichlet de la bola unitaria

En dimensión tres, la separación produce funciones radiales

$$
j_\ell(\sqrt\lambda\,r).
$$

La condición de Dirichlet en $r=1$ exige

$$
j_\ell(\sqrt\lambda)=0.
$$

Si $\alpha_{\ell,k}$ es el $k$-ésimo cero positivo de $j_\ell$, entonces

$$
\lambda_{\ell,k}
=
\alpha_{\ell,k}^2.
$$

Cada valor asociado a $\ell$ tiene multiplicidad angular $2\ell+1$.

In [ ]:
# ============================================================
# ESPECTRO DE LA BOLA
# ============================================================

l_max = 6
k_max = 5

spectral_data = []

for l in range(l_max + 1):
    zeros = positive_spherical_bessel_zeros(
        l,
        k_max,
    )

    for k, zero in enumerate(zeros, start=1):
        spectral_data.append(
            {
                "l": l,
                "k": k,
                "zero": float(zero),
                "lambda": float(zero**2),
                "multiplicity": 2 * l + 1,
            }
        )

spectral_data_sorted = sorted(
    spectral_data,
    key=lambda item: item["lambda"],
)

for index, item in enumerate(
    spectral_data_sorted[:15],
    start=1,
):
    print(
        index,
        item,
    )

fig, ax = plt.subplots(figsize=(9, 5.8))

for item in spectral_data:
    ax.scatter(
        item["l"],
        item["lambda"],
        s=25 + 8 * item["multiplicity"],
    )

ax.set_xlabel(r"$\ell$")
ax.set_ylabel(r"$\lambda_{\ell,k}$")
ax.set_title("Valores propios de la bola unitaria")
ax.grid(True, alpha=0.3)
fig.tight_layout()

path = FIG_DIR / "03.9.D_espectro_bola.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print(path.resolve())

In [ ]:
# Primeros perfiles radiales normalizados.

r = np.linspace(0.0, 1.0, 1200)

profiles_to_plot = [
    (0, 1),
    (1, 1),
    (2, 1),
    (0, 2),
]

fig, ax = plt.subplots(figsize=(9, 5.8))

for l, k in profiles_to_plot:
    zero = positive_spherical_bessel_zeros(
        l,
        k,
    )[-1]
    profile = spherical_jn(l, zero * r)
    profile = profile / np.max(np.abs(profile))

    ax.plot(
        r,
        profile,
        label=rf"$(\ell,k)=({l},{k})$",
    )

ax.axhline(0.0, linewidth=1.0)
ax.set_xlabel(r"$r$")
ax.set_ylabel("perfil radial normalizado")
ax.set_title("Funciones radiales de Dirichlet")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

path = FIG_DIR / "03.9.D_perfiles_radiales.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print(path.resolve())

## Simulación 3.9.E — Resonancia

Sea $\phi_1$ una función propia normalizada con valor propio $\lambda_1$. Para

$$
(-\Delta-\mu)u=\phi_1,
$$

la solución formal fuera de resonancia es

$$
u_\mu
=
\frac{1}{\lambda_1-\mu}\phi_1.
$$

Su norma diverge cuando $\mu\to\lambda_1$. En resonancia exacta no existe solución
porque el dato no es ortogonal al núcleo.

In [ ]:
lambda_first = spectral_data_sorted[0]["lambda"]

mu_left = np.linspace(
    max(0.0, lambda_first - 16.0),
    lambda_first - 0.03,
    600,
)
mu_right = np.linspace(
    lambda_first + 0.03,
    lambda_first + 16.0,
    600,
)
mu_values = np.concatenate(
    [mu_left, mu_right]
)

response_norm = 1.0 / np.abs(
    lambda_first - mu_values
)

fig, ax = plt.subplots(figsize=(9, 5.8))
ax.semilogy(
    mu_values,
    response_norm,
)
ax.axvline(
    lambda_first,
    linestyle="--",
    label=rf"$\lambda_1={lambda_first:.6f}$",
)
ax.set_xlabel(r"$\mu$")
ax.set_ylabel(r"$\|u_\mu\|$")
ax.set_title("Amplificación resonante")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

path = FIG_DIR / "03.9.E_resonancia.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print(path.resolve())

# 3.9.1 Problema espectral de Dirichlet

## **Definición 3.9.1 (Valor propio y función propia).**

Sea $\Omega\subset\mathbb R^n$ un dominio acotado. Un número
$\lambda\in\mathbb R$ es un **valor propio de Dirichlet de $-\Delta$** si existe

$$
u\not\equiv0
$$

tal que

$$
\begin{cases}
-\Delta u=\lambda u,
& \text{en }\Omega,\\
u=0,
& \text{sobre }\partial\Omega.
\end{cases}
$$

La función $u$ se llama **función propia** asociada a $\lambda$.

## **Proposición 3.9.2 (Positividad de los valores propios).**

Todo valor propio de Dirichlet satisface

$$
\lambda>0.
$$

### Demostración

Multiplicamos la ecuación por $u$ e integramos. La identidad de Green y el dato
de frontera dan

$$
\int_\Omega|\nabla u|^2\,dx
=
\lambda
\int_\Omega u^2\,dx.
$$

Como $u\not\equiv0$, el lado derecho obliga a $\lambda\geq0$. Si
$\lambda=0$, entonces $\nabla u=0$ y $u$ es constante; el dato de frontera
implica $u=0$, contradicción.

$\square$

## **Definición 3.9.3 (Cociente de Rayleigh).**

Para $v\in H_0^1(\Omega)$, $v\neq0$, se define

$$
\mathcal R(v)
=
\frac{
\displaystyle\int_\Omega|\nabla v|^2\,dx
}{
\displaystyle\int_\Omega v^2\,dx
}.
$$

## **Teorema 3.9.4 (Caracterización variacional del primer valor propio).**

El primer valor propio satisface

$$
\lambda_1
=
\inf_{\substack{
v\in H_0^1(\Omega)\\
v\neq0
}}
\mathcal R(v).
$$

El ínfimo se alcanza en una función propia $\phi_1$.

### Demostración añadida

Se minimiza la energía

$$
\int_\Omega|\nabla v|^2\,dx
$$

sobre la restricción

$$
\int_\Omega v^2\,dx=1.
$$

Una sucesión minimizante es acotada en $H_0^1$. Por compacidad débil y la
inmersión compacta de $H_0^1$ en $L^2$ para dominios acotados, el límite conserva
la normalización. La semicontinuidad inferior da un minimizador.

La ecuación de Euler–Lagrange con multiplicador $\lambda_1$ es

$$
\int_\Omega
\nabla\phi_1\cdot\nabla\varphi\,dx
=
\lambda_1
\int_\Omega\phi_1\varphi\,dx.
$$

Por tanto, $\phi_1$ es una función propia.

$\square$

## **Proposición 3.9.5 (Ortogonalidad).**

Sean $u$ y $v$ funciones propias asociadas a valores distintos
$\lambda\neq\mu$. Entonces

$$
\int_\Omega uv\,dx=0.
$$

### Demostración

Usando las dos ecuaciones y la identidad de Green,

$$
\int_\Omega
\nabla u\cdot\nabla v\,dx
=
\lambda\int_\Omega uv\,dx,
$$

y también

$$
\int_\Omega
\nabla u\cdot\nabla v\,dx
=
\mu\int_\Omega uv\,dx.
$$

Por tanto,

$$
(\lambda-\mu)
\int_\Omega uv\,dx=0.
$$

$\square$

### Ejercicios — Sección 3.9.1

> **Ruta corta de estudio:** dos ejercicios de técnica y una variación de nivel Examen General.

1. Demuestre que el primer valor propio es simple en un dominio conexo.

2. Pruebe que una primera función propia puede elegirse estrictamente positiva.

3. **Tipo Examen General.** Sea $c<0$. Use una identidad de energía para demostrar
   que la única solución de

   $$
   \Delta u+cu=0,
   \qquad
   u|_{\partial\Omega}=0,
   $$

   es la solución trivial.


# 3.9.2 Laplaciano en coordenadas esféricas

## **Proposición 3.9.6 (Descomposición radial-angular).**

Para una función suave $u=u(r,\omega)$, con

$$
r=|x|,
\qquad
\omega=\frac{x}{|x|}\in\mathbb S^{n-1},
$$

el Laplaciano se escribe

$$
\Delta u
=
\frac{\partial^2u}{\partial r^2}
+
\frac{n-1}{r}
\frac{\partial u}{\partial r}
+
\frac{1}{r^2}
\Delta_{\mathbb S^{n-1}}u.
$$

El operador $\Delta_{\mathbb S^{n-1}}$ es el
**Laplaciano de Laplace–Beltrami** sobre la esfera.

En dimensión tres,

$$
\Delta_{\mathbb S^2}
=
\frac{1}{\sin\theta}
\frac{\partial}{\partial\theta}
\left(
\sin\theta
\frac{\partial}{\partial\theta}
\right)
+
\frac{1}{\sin^2\theta}
\frac{\partial^2}{\partial\varphi^2}.
$$

## **Definición 3.9.7 (Armónico esférico).**

Un **armónico esférico de grado $\ell$** es una función no nula
$Y$ sobre $\mathbb S^{n-1}$ que satisface

$$
-\Delta_{\mathbb S^{n-1}}Y
=
\ell(\ell+n-2)Y,
\qquad
\ell=0,1,2,\dots.
$$

En $\mathbb S^2$,

$$
-\Delta_{\mathbb S^2}Y_\ell^m
=
\ell(\ell+1)Y_\ell^m,
$$

con

$$
m=-\ell,-\ell+1,\dots,\ell.
$$

La multiplicidad es

$$
2\ell+1.
$$

## **Teorema 3.9.8 (Ortogonalidad en la esfera).**

Los armónicos esféricos asociados a valores propios distintos son ortogonales en

$$
L^2(\mathbb S^{n-1}).
$$

En $\mathbb S^2$, puede elegirse una base ortonormal

$$
\{Y_\ell^m:-\ell\leq m\leq\ell\}.
$$

## **Proposición 3.9.9 (Relación con polinomios armónicos homogéneos).**

Sea $P_\ell$ un polinomio homogéneo de grado $\ell$:

$$
P_\ell(rx)=r^\ell P_\ell(x).
$$

Si

$$
\Delta P_\ell=0
$$

en $\mathbb R^n$, entonces su restricción

$$
Y(\omega)=P_\ell(\omega)
$$

a la esfera es un armónico esférico de grado $\ell$.

Recíprocamente, si $Y$ es un armónico esférico de grado $\ell$, entonces

$$
P(x)
=
|x|^\ell
Y\left(\frac{x}{|x|}\right)
$$

es un polinomio armónico homogéneo.

### Demostración

Aplicando la descomposición del Laplaciano a

$$
u(r,\omega)=r^\ell Y(\omega),
$$

se obtiene

$$
\Delta u
=
r^{\ell-2}
\left[
\ell(\ell+n-2)Y
+
\Delta_{\mathbb S^{n-1}}Y
\right].
$$

La expresión se anula exactamente cuando $Y$ satisface la ecuación de
Laplace–Beltrami.

$\square$

### Ejercicios — Sección 3.9.2

> **Ruta corta de estudio:** dos ejercicios de técnica y una variación de nivel Examen General.

1. Verifique que las restricciones de $x_1,\dots,x_n$ a la esfera son armónicos
   de grado uno.

2. Determine cuáles de los polinomios

   $$
   x^2-y^2,
   \qquad
   xy,
   \qquad
   x^2+y^2-2z^2
   $$

   son armónicos homogéneos.

3. **Tipo Examen General.** Deduzca la ecuación de Laplace–Beltrami mediante
   separación de variables en coordenadas esféricas.


# 3.9.3 Primeros modos en $\mathbb S^2$

## **Proposición 3.9.10 (Primeros armónicos esféricos reales).**

Salvo constantes de normalización:

### Grado $\ell=0$

$$
1.
$$

### Grado $\ell=1$

$$
x,
\qquad
y,
\qquad
z
$$

restringidos a $\mathbb S^2$.

### Grado $\ell=2$

$$
xy,
\qquad
xz,
\qquad
yz,
$$

$$
x^2-y^2,
\qquad
2z^2-x^2-y^2.
$$

Estos cinco modos corresponden a la multiplicidad

$$
2\ell+1=5.
$$

### Interpretación geométrica

- $\ell=0$ no tiene nodos.
- $\ell=1$ tiene un gran círculo nodal.
- $\ell=2$ posee patrones cuadrupolares.
- En general, el grado controla la complejidad angular.

# 3.9.4 Separación radial para funciones armónicas

## **Proposición 3.9.11 (Soluciones separadas de Laplace).**

Busquemos

$$
u(r,\omega)=R(r)Y_\ell(\omega)
$$

con $Y_\ell$ de grado $\ell$. La ecuación $\Delta u=0$ conduce a

$$
r^2R''+(n-1)rR'
-
\ell(\ell+n-2)R=0.
$$

Las soluciones son

$$
R(r)=Ar^\ell+Br^{-(\ell+n-2)}
$$

para $n\geq3$.

En dimensión dos, para $\ell\geq1$,

$$
R(r)=Ar^\ell+Br^{-\ell},
$$

mientras que el modo $\ell=0$ tiene soluciones

$$
R(r)=A+B\log r.
$$

### Consecuencia

En una bola que contiene al origen, la regularidad elimina la rama singular y

$$
u(r,\omega)
=
\sum_{\ell,m}
a_{\ell m}r^\ell Y_\ell^m(\omega).
$$

### Ejercicios — Secciones 3.9.3 y 3.9.4

> **Ruta corta de estudio:** dos ejercicios de técnica y una variación de nivel Examen General.

1. Obtenga la expansión armónica de un dato de frontera que sea combinación de
   modos de grado uno y dos.

2. Explique por qué la rama singular puede aparecer en un problema exterior.

3. **Tipo Examen General.** Encuentre todas las funciones armónicas con simetría
   esférica en $\mathbb R^3\setminus\{0\}$.


# 3.9.5 Valores propios de la bola

## **Teorema 3.9.12 (Separación para el problema espectral en la bola).**

Sea

$$
B_R=\{x\in\mathbb R^3:|x|<R\}.
$$

Para el problema

$$
\begin{cases}
-\Delta u=\lambda u,
& \text{en }B_R,\\
u=0,
& \text{sobre }\partial B_R,
\end{cases}
$$

buscamos

$$
u(r,\omega)=R_\ell(r)Y_\ell^m(\omega).
$$

La ecuación radial es

$$
r^2R_\ell''
+
2rR_\ell'
+
\left(
\lambda r^2-\ell(\ell+1)
\right)R_\ell
=
0.
$$

La solución regular en el origen es

$$
R_\ell(r)
=
j_\ell(\sqrt\lambda\,r),
$$

donde $j_\ell$ es la función de Bessel esférica.

La condición de frontera exige

$$
j_\ell(\sqrt\lambda\,R)=0.
$$

Si $\alpha_{\ell,k}$ es el $k$-ésimo cero positivo de $j_\ell$, entonces

$$
\boxed{
\lambda_{\ell,k}
=
\frac{\alpha_{\ell,k}^2}{R^2}.
}
$$

La multiplicidad angular es $2\ell+1$.

## **Corolario 3.9.13 (Modo radial).**

Para $\ell=0$,

$$
j_0(s)=\frac{\sin s}{s}.
$$

Por tanto, los valores propios radiales son

$$
\lambda_{0,k}
=
\frac{k^2\pi^2}{R^2}.
$$

Las funciones propias radiales son, salvo normalización,

$$
u_k(r)
=
\frac{\sin(k\pi r/R)}{r}.
$$

El valor en $r=0$ se interpreta por continuidad.

### Ejercicios — Sección 3.9.5

> **Ruta corta de estudio:** dos ejercicios de técnica y una variación de nivel Examen General.

1. Derive la ecuación de Bessel esférica desde la separación.

2. Verifique la fórmula del modo radial resolviendo directamente

   $$
   u''+\frac{2}{r}u'+\lambda u=0.
   $$

3. **Tipo Examen General.** Muestre que para ciertos $c>0$ existen soluciones no
   triviales de

   $$
   \Delta u+cu=0
   $$

   en una bola con dato de Dirichlet cero.


# 3.9.6 Resonancia y alternativa de Fredholm

## **Teorema 3.9.14 (Condición necesaria de compatibilidad).**

Sea $\lambda$ un valor propio de Dirichlet y sea $\phi$ una función propia:

$$
-\Delta\phi=\lambda\phi,
\qquad
\phi|_{\partial\Omega}=0.
$$

Si existe una solución de

$$
\begin{cases}
(-\Delta-\lambda)u=f,
& \text{en }\Omega,\\
u=0,
& \text{sobre }\partial\Omega,
\end{cases}
$$

entonces necesariamente

$$
\int_\Omega f\phi\,dx=0.
$$

Más generalmente, $f$ debe ser ortogonal a todo el espacio propio asociado a
$\lambda$.

### Demostración

Multiplicamos la ecuación de $u$ por $\phi$ e integramos:

$$
\int_\Omega
\left(
-\Delta u-\lambda u
\right)\phi\,dx
=
\int_\Omega f\phi\,dx.
$$

Por la identidad de Green y los datos de Dirichlet,

$$
\int_\Omega
u\left(
-\Delta\phi-\lambda\phi
\right)dx
=
\int_\Omega f\phi\,dx.
$$

El miembro izquierdo es cero.

$\square$

## **Teorema 3.9.15 (Alternativa espectral, versión formal).**

Para el Laplaciano de Dirichlet en un dominio acotado y suficientemente regular:

1. si $\lambda$ no es valor propio, el problema
   $$
   (-\Delta-\lambda)u=f
   $$
   tiene una única solución débil;
2. si $\lambda$ es valor propio, existe solución si y sólo si $f$ es ortogonal
   al espacio propio;
3. cuando existe solución en resonancia, no es única: puede sumarse cualquier
   función propia del núcleo.

### **Aclaración.**

La demostración funcional completa usa la teoría espectral compacta del inverso
de Dirichlet. En este notebook se demuestra con detalle la condición necesaria
y se interpreta mediante expansiones en funciones propias.

## **Ejemplo 3.9.16 (Bola de radio $\pi$).**

Sea

$$
\Omega=B_\pi(0)\subset\mathbb R^3.
$$

La función radial

$$
\phi(x)
=
\frac{\sin|x|}{|x|}
$$

satisface

$$
-\Delta\phi=\phi
$$

en la bola y

$$
\phi=0
$$

sobre $|x|=\pi$.

Por tanto, para que exista una solución de

$$
-\Delta u-u=f,
\qquad
u|_{\partial B_\pi}=0,
$$

es necesario que

$$
\int_{B_\pi}
f(x)
\frac{\sin|x|}{|x|}
\,dx
=
0.
$$

Este es exactamente el mecanismo de compatibilidad espectral que aparece en el
Examen General 2025-1.

### Ejercicios — Sección 3.9.6

> **Ruta corta de estudio:** dos ejercicios de técnica y una variación de nivel Examen General.

1. Demuestre que en resonancia la solución, cuando existe, no es única.

2. Escriba la solución formal en una base ortonormal de funciones propias cuando
   $\lambda$ no pertenece al espectro.

3. **Tipo Examen General.** Sea $\Omega=B_\pi(0)\subset\mathbb R^3$.
   Deduzca una condición necesaria de compatibilidad para

   $$
   \Delta u+u=f,
   \qquad
   u|_{\partial\Omega}=0.
   $$


# Control de cobertura y estado del capítulo

## Contenido cubierto

- problema espectral de Dirichlet;
- positividad de los valores propios;
- cociente de Rayleigh;
- primer valor propio;
- ortogonalidad;
- Laplaciano esférico;
- Laplace–Beltrami;
- armónicos esféricos;
- primeros modos;
- polinomios armónicos homogéneos;
- separación radial para Laplace;
- valores propios de la bola;
- funciones de Bessel esféricas;
- resonancia y compatibilidad;
- visualizaciones tridimensionales y espectrales.

## Relación con las fuentes

La unidad completa un hueco explícito del temario oficial. Las notas manuscritas
disponibles no contienen este bloque. La sección de resonancia se alineó con los
problemas 4 y 5 del Examen General 2025-1.

## Pendiente inmediato

El siguiente y último notebook de la cola es

$$
\texttt{03.10.01\_Aplicaciones\_y\_simulacro\_integrador.ipynb}.
$$

Se integrarán membranas, electrostática y flujo potencial, junto con un
simulacro final de nivel Examen General.